<a target="_blank" href="https://colab.research.google.com/github/novainsilico/jinko-api-cookbook/blob/main/tutorials/02-api_onboarding_Influenza.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Module 6 : Leveraging Jinkō’s API for Programmatic Access - Influenza Use case

### Introduction
Goal of this cookbook is to illustrate how one can post the material needed for a trial, run it and visualize the results.

### Steps:
1. Post the Computational Model
2. Create a Virtual Population (Vpop) Design
3. Generate a Vpop from the Vpop Design
4. Post a Protocol
5. Post an Output set
6. Post a Trial
7. Run and monitor a Trial
8. Visualize the trial results
9. Hands-on!

### Linked resources:
- [API tutorials](https://doc.jinko.ai/docs/category/api---tutorials/): guidelines and examples on how to use the SDK
- [API Reference](https://doc.jinko.ai/api/#/): exhaustive documentation of jinkō's API
- [Public cookbook repository](https://github.com/novainsilico/jinko-api-cookbook/tree/main): cookbooks examples of use cases we compiled
- [jinko.ai](https://jinko.ai): jinkō website, you'll need an account

### What you'll need
- A Jinko API key with access to your project.
- The project id (UUID).
- Python environment: [jinko-sdk](https://pypi.org/project/jinko-sdk/) package installed. In Colab, it's installed via the notebook.
- Optional: set `JINKO_API_KEY` and `JINKO_PROJECT_ID` via `.envrc` (see `.envrc.sample`).

---

## Step 0 : Connect to jinkō and setup the cookbook

In [ ]:
import json
import os
import sys
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
if "google.colab" in sys.modules:
    !pip install -q jinko-sdk

from jinko import JinkoClient


### Connection initialization
- See the [Getting Started](https://github.com/novainsilico/jinko-api-cookbook/blob/main/tutorials/01-getting_started.ipynb) notebook for fuller setup instructions.
- Create an API key for the project [here](https://jinko.ai/project/27e8eba0-a869-4c5d-b3b3-417db5a6e186/settings) and store it in your secrets (`.envrc` locally or Colab userdata).
- The project id is the UUID in the project URL.

In [ ]:
# Load secrets from Colab (if running in Colab)
if "google.colab" in sys.modules:
    from google.colab import userdata

    api_key = userdata.get("JINKO_API_KEY")
    project_id = userdata.get("JINKO_PROJECT_ID")
    if api_key:
        os.environ["JINKO_API_KEY"] = api_key
        print("JINKO_API_KEY is set")
    if project_id:
        os.environ["JINKO_PROJECT_ID"] = project_id
        print("JINKO_PROJECT_ID is set")
else:
    api_key = None
    project_id = None

client = JinkoClient()
client.auth_check()

constants of the cookbook:

In [ ]:
# FOLDER_ID can be retrieved in the url of the folder, pattern is `https://jinko.ai/project/<project_id>?labels=<folder_id>`
FOLDER_ID = ""  # TO BE PROVIDED

# Step 1 : Post the Computational Model

We will use the Influenza model, available in this repo

In [ ]:
# Model Retrieval - do not modify

resources_dir = Path("./resources")
MODEL_FILE_LOCAL = resources_dir / "CM_Influenza.json"
SOLVING_OPTIONS_FILE_LOCAL = resources_dir / "SolvingOptions_Influenza.json"

MODEL_URL = "https://raw.githubusercontent.com/novainsilico/jinko-api-cookbook/main/tutorials/resources/CM_Influenza.json"
SOLVING_OPTIONS_URL = "https://raw.githubusercontent.com/novainsilico/jinko-api-cookbook/main/tutorials/resources/SolvingOptions_Influenza.json"


def load_json(local_path: Path, remote_url: str) -> dict:
    try:
        # local-first
        with local_path.open("r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        # fallback
        print(f"Fetching {local_path.name} from GitHub…")
        local_path.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(remote_url, local_path)
        with local_path.open("r", encoding="utf-8") as f:
            return json.load(f)


model = load_json(MODEL_FILE_LOCAL, MODEL_URL)
solving_options = load_json(SOLVING_OPTIONS_FILE_LOCAL, SOLVING_OPTIONS_URL)

In [ ]:
# ---- Post the model with its options ----
payload = {"model": model, "solvingOptions": solving_options}

model = client.create_model_from_json(
    json_content=payload,
    name="CM Influenza",
    folder=FOLDER_ID or None,
)

print(f"Resource link: {model.url}")

# Step 2 : Create a Vpop Design

Let's retrieve the patient descriptors from the model and define their distributions for the Vpop design creation.

In [ ]:
# Retrieve baseline descriptors through the typed model helper.
numeric_descriptors = model.get_baseline_descriptors().model_dump(by_alias=True)[
    "numericDescriptors"
]

patient_descriptors = [
    descriptor["id"]
    for descriptor in numeric_descriptors
    if any(
        tag in descriptor["inputTag"]
        for tag in [
            "PatientDescriptorKnown",
            "PatientDescriptorUnknown",
            "PatientDescriptorPartiallyKnown",
        ]
    )
]

print(f"{len(patient_descriptors)} patient descriptors:")
display(patient_descriptors)

Create a new list with the updated distributions:

In [ ]:
patient_descriptors_distribution = [
    {
        "distribution": {"mean": 8e-6, "stdev": 8e-7, "tag": "Normal"},
        "id": "beta",
    },
    {
        "distribution": {"highBound": 1.02, "lowBound": 0.68, "tag": "Uniform"},
        "id": "d",
    },
    {
        "distribution": {"highBound": 1.92, "lowBound": 1.28, "tag": "Uniform"},
        "id": "kappa",
    },
    {
        "distribution": {"highBound": 9.24e-05, "lowBound": 6.16e-05, "tag": "Uniform"},
        "id": "p",
    },
    {
        "distribution": {"highBound": 0.0828, "lowBound": 0.0552, "tag": "Uniform"},
        "id": "phi",
    },
    {
        "distribution": {"highBound": 7.32e-10, "lowBound": 4.88e-10, "tag": "Uniform"},
        "id": "q",
    },
    {
        "distribution": {"highBound": 0.012, "lowBound": 0.008, "tag": "Uniform"},
        "id": "rho",
    },
]

post the virtual population design:

In [ ]:
vpop_design = model.create_vpop_design_from_design(
    marginal_distributions={
        item["id"]: item["distribution"] for item in patient_descriptors_distribution
    },
    name="vpop design for Influenza model",
    folder=FOLDER_ID or None,
)

print(f"Virtual Population Design Resource link: {vpop_design.url}")

# Step 3 : Generate a Vpop from the Vpop design

From the virtual population design, we generate a virtual population of 100 patients

In [ ]:
vpop = vpop_design.generate_vpop(
    size=100,
    seed=42,
    variance_reduction=True,
    name="vpop for Influenza model",
    folder=FOLDER_ID or None,
)

print(f"Virtual Population Resource link: {vpop.url}")

# Step 4 : Post a Protocol

We are posting a protocol with 2 arms:
 - Default_arm
 - Double_c

In [ ]:
scenario_arms = [
    {
        "armControl": None,
        "armIsActive": True,
        "armName": "Default_arm",
        "armOverrides": [
            {"formula": "20.0", "key": "c"},
        ],
        "armWeight": 1,
    },
    {
        "armControl": None,
        "armIsActive": True,
        "armName": "Double_c",
        "armOverrides": [
            {"formula": "40.0", "key": "c"},
        ],
        "armWeight": 1,
    },
]

In [ ]:
protocol = model.create_protocol_design(
    scenario_arms,
    name="protocol for Influenza model",
    folder=FOLDER_ID or None,
)

print(f"Protocol Resource link: {protocol.url}")

# Step 5 : Post a Simple Output Set

We define which time series we want to observe in the simulation


In [ ]:
measures = ["V", "F", "I"]

In [ ]:
simple_output_set = client.create_simple_output_set(
    model,
    measures,
    name="simple output set for Influenza model",
    folder=FOLDER_ID or None,
)

print(f"Simple Output Set Resource link: {simple_output_set.url}")

# Step 6 : Post a Trial

In [ ]:
trial = model.create_trial(
    vpop=vpop,
    protocol=protocol,
    simple_output_set=simple_output_set,
    name="trial for Influenza model",
    folder=FOLDER_ID or None,
)

print(f"Trial Resource link: {trial.url}")

# Step 7 : Run and monitor a trial

We are running the trial simulation

In [ ]:
trial.run()

waiting for the trial to complete


In [ ]:
trial.wait_until_completed()
display(trial.status())

# Step 8 : Visualize the trial results

Once the trial is completed, we can download and visualize the results

In [ ]:
results_summary = trial.results.summary()
arm_names = results_summary["arms"]

display("Available time series:", trial.output_ids())

download the time series:

In [ ]:
TIME_SERIES_IDS = ["V"]

print("Retrieving time series data...")
dfTimeSeries = trial.results.timeseries(
    {timeseries_id: arm_names for timeseries_id in TIME_SERIES_IDS}
).to_dataframe()
print("Time series data retrieved successfully.")

load the time series in a dataframe:

In [ ]:
display(dfTimeSeries.head(5))

unique_patient_ids = dfTimeSeries["Patient Id"].unique().tolist()

plot the time series of the first patient:

In [ ]:
patient_data = dfTimeSeries[dfTimeSeries["Patient Id"]
                            == unique_patient_ids[0]]

for arm, group in patient_data.groupby("Arm"):
    plt.plot(group["Time"], group["Value"],
             marker="o", linestyle="-", label=arm)

plt.title("Time Series of Viral load")
plt.xlabel("Time (seconds)")
plt.ylabel("V")
plt.legend(title="Arm")

plt.show()

# Hands On part:

In the following cells:

 - create a vpop design with different distribution types. Generate a vpop of 50 patients.
 - create a protocol with 20 arms where c Dose varies from 10 to 100
 - post a new trial with this vpop and protocol 
 - run the trial
 - plot the time series you want, tune the plot

 Feel free to explore cases using the doc or the cookbooks repository.

 Have fun :)